In [1]:
import sys
import numpy as np
import torch
from torch import nn

# 你的脚本所在目录
sys.path.append(r"C:\git-workplace\HeatSim\HypridMethod")

from mionet_linear import LinearMIONet, MIONetConfig

# 1. 路径
data_dir = r"C:\git-workplace\HeatSim\HypridMethod\Data\data\mionet_gp_2d_5000"
ckpt_path = r"C:\git-workplace\HeatSim\HypridMethod\Train1\best.pt"


# 2. 读取数据形状，用来确定 k_dim 和 f_dim
k_sensor = np.load(data_dir + r"\k_sensor.npy", mmap_mode="r")
f_sensor = np.load(data_dir + r"\f_sensor.npy", mmap_mode="r")
output_points = np.load(data_dir + r"\output_points.npy")

k_dim = k_sensor.shape[1]
f_dim = f_sensor.shape[1]

print("k_dim:", k_dim)
print("f_dim:", f_dim)
print("output_points:", output_points.shape)


# 3. 重新创建和训练时一样的模型
cfg = MIONetConfig(
    k_dim=k_dim,
    f_dim=f_dim,
    coord_dim=2,
    latent_dim=500,
    k_hidden=(500, 500, 500),
    trunk_hidden=(500, 500, 500),
    activation=nn.Tanh,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LinearMIONet(cfg).to(device)


# 4. 读取 best.pt
ckpt = torch.load(ckpt_path, map_location=device)

print("checkpoint keys:", ckpt.keys())
print("best epoch:", ckpt["epoch"])
print("metrics:", ckpt["metrics"])


# 5. 加载最优模型参数
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("best.pt loaded successfully.")

k_dim: 10000
f_dim: 10000
output_points: (10201, 2)
checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'config', 'metrics'])
best epoch: 690
metrics: {'train_mse': 1.408242657958504e-05, 'train_rel_l2': 0.17648979936705694, 'val_mse': 1.307031902797462e-05, 'val_rel_l2': 0.17170404344797136, 'best_val_rel_l2': 0.17170404344797136}
best.pt loaded successfully.


C:\Users\Rainbow\AppData\Local\Temp\ipykernel_10140\1645311132.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


In [2]:
sample_id = 220

k = torch.tensor(k_sensor[sample_id:sample_id+1], dtype=torch.float32, device=device)
f = torch.tensor(f_sensor[sample_id:sample_id+1], dtype=torch.float32, device=device)
coords = torch.tensor(output_points, dtype=torch.float32, device=device)

with torch.no_grad():
    u_pred = model(k, f, coords)

u_pred = u_pred.detach().cpu().numpy().reshape(-1)

print("u_pred shape:", u_pred.shape)
print("u_pred min/max:", u_pred.min(), u_pred.max())

u_pred shape: (10201,)
u_pred min/max: -0.033144522 0.0032908134


In [3]:
u_train = np.load(data_dir + r"\solved_5000\u_train.npy", mmap_mode="r")

u_true = u_train[sample_id]

rel_l2 = np.linalg.norm(u_pred - u_true) / np.linalg.norm(u_true)

print("relative L2 error:", rel_l2)

relative L2 error: 0.22226422
